# Deep Agent Lab
Follow the lifecycle of an evidence brief from retrieval to review. The exercises use the real Deep Agents harness, a small fictional corpus, and scripted model responses. Run top to bottom. The optional live cell uses Ollama and the course's PostgreSQL/OpenSearch corpus.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from deep_agent_lab.labs import (
    async_lab,
    context_lab,
    delegation_lab,
    interpreter_lab,
    mcp_lab,
    memory_lab,
    shell_lab,
    summarization_lab,
    workspace_lab,
)
from deep_agent_lab.service import (
    AgentService,
    Decision,
    ReviewRequest,
    RunRequest,
    open_checkpointer,
)
from deep_agent_lab.settings import Settings

scratch = TemporaryDirectory()
settings = Settings(
    _env_file=None, deep_lab_mode="demo",
    deep_lab_state_dir=Path(scratch.name), langfuse_enabled=False,
)

## 1. Tools, MCP, and threads
The model issues a search call, a tool returns passages, and the answer cites one or more exact evidence tokens. Each thread holds its own history.

In [2]:
with open_checkpointer(settings) as checkpoints:
    service = AgentService(settings, checkpoints)
    try:
        answer = service.run(RunRequest(
            prompt="Who approves a production restore?", thread_id="lesson-1",
        ))
    finally:
        service.close()

assert "database operations lead" in answer.answer
assert answer.evidence[0].citation in answer.answer
assert not answer.unknown_citations
print(answer.answer)
print(answer.steps)

## Findings

- The database operations lead approves production restores. Restore into an isolated database first and validate the requested records before touching production. After restoration, designated operators must verify service health and record the validation result. [[bfa9b1f0bc3e]]

## Next steps

Verify the source excerpts.
['search_knowledge']


In [3]:
mcp = await mcp_lab()
assert "search" in mcp["tools"]
assert "northstar-restore" in mcp["results"][0]
print(mcp["tools"])

['search']


## 2. Execution environment
Route reference paths to a temporary directory and draft paths to graph state. Compare a single fixed shell example with a QuickJS calculation.

In [4]:
workspace = workspace_lab()
assert "permission denied" in workspace["results"][1]
assert "/draft.md" in workspace["state_files"]
print(workspace)
print(shell_lab())
print(interpreter_lab())

{'results': ['1  Original policy.', 'Error: permission denied for write on /reference/alternate.md', 'Updated file /draft.md', '1  A working draft.'], 'host_files': ['policy.md'], 'state_files': ['/draft.md']}
{'command': 'type policy.txt', 'exit_code': 0, 'output': 'Restore approval recorded.'}


C:\Users\imrah\Documents\deepagents\src\deep_agent_lab\labs.py:134: LangChainBetaWarning: The class `CodeInterpreterMiddleware` is in beta. It is actively being worked on, so the API may change.
  middleware=[CodeInterpreterMiddleware(subagents=False, timeout=2)],


{'results': ['<stdout>\n90\n</stdout>\n<result>null</result>']}


## 3. Context, skills, and memory
A long tool result gets a path and preview. Compaction creates a separate history file. Shared project memory and thread state have different lifetimes.

In [5]:
offload = context_lab()
summary = summarization_lab()
assert offload["files"][0].startswith("/large_tool_results/")
assert summary["files"] == ["/conversation_history/summary-lab.md"]
print(offload["files"], summary["files"])

memory = memory_lab()
assert memory["A"]["files"] == ["/private.md"]
assert memory["B"]["files"] == []
print(memory["A"]["tool_results"][0])

['/large_tool_results/fetch_archive'] ['/conversation_history/summary-lab.md']
1  # Brief format
2  Use short bullets.


## 4. Plan and delegate
QuickJS dispatches two specialists in parallel. The main graph receives two short reports and updates its plan. Try adding a third topic in the lab source.

In [6]:
delegation = await delegation_lab()
assert "incident commander" in delegation["results"][1]
assert "database operations lead" in delegation["results"][1]
assert delegation["todos"][0]["status"] == "completed"
print(delegation["results"][1])

<stdout>
["The database operations lead approves restores.","The incident commander coordinates the response."]
</stdout>
<result>null</result>


## 5. Capstone
Draft a brief, observe the skill read and approval request, then close the checkpoint connection. Reopen the store before approving. This simulates saving after a CLI restart.

In [7]:
with open_checkpointer(settings) as checkpoints:
    service = AgentService(settings, checkpoints)
    try:
        proposal = service.run(RunRequest(
            prompt="Draft and save an evidence brief about production restore approval.",
            thread_id="capstone",
        ))
    finally:
        service.close()

assert proposal.status == "interrupted"
assert "read_file" in proposal.steps
print(proposal.reviews[0].action_requests[0]["args"]["markdown"])

## Findings

- The database operations lead approves production restores. Restore into an isolated database first and validate the requested records before touching production. After restoration, designated operators must verify service health and record the validation result. [[bfa9b1f0bc3e]]

## Next steps

Verify the source excerpts.


In [8]:
with open_checkpointer(settings) as checkpoints:
    service = AgentService(settings, checkpoints)
    try:
        result = service.resume("capstone", ReviewRequest(
            interrupt_id=proposal.reviews[0].id,
            decisions=[Decision(type="approve")],
        ))
        brief_id = result.saved_brief_ids[0]
        saved = service.store.get(brief_id)
        assert saved and saved.thread_id == "capstone"
        print(saved.title, saved.id)
    finally:
        service.close()

print(async_lab())
scratch.cleanup()

Northstar evidence brief 94bb5178e3f1f28ed8aa088b5760be4c
{'worker_running': True, 'supervisor': 'The brief outline is ready while research runs.', 'worker_answer': 'The database operations lead approves a production restore.'}


## 6. Managed deployment
Open [the managed project walkthrough](../docs/managed-deployment.md). Sketch how the managed runtime would reach your search service and which store would own approved briefs. The deployment exercise is a design review.

### Optional live run
Initialize and ingest the course corpus, check `deep-lab doctor`, then set `RUN_LIVE` to `True`. This query reads evidence from your existing corpus.

In [9]:
RUN_LIVE = False
if RUN_LIVE:
    root = Path.cwd()
    if root.name == "notebooks":
        root = root.parent
    live = Settings(_env_file=root / ".env", deep_lab_mode="live")
    if not live.deep_lab_state_dir.is_absolute():
        live.deep_lab_state_dir = root / live.deep_lab_state_dir
    with open_checkpointer(live) as checkpoints:
        service = AgentService(live, checkpoints)
        try:
            result = service.run(RunRequest(prompt="Who approves a production restore?"))
            print(result.model_dump_json(indent=2))
        finally:
            service.close()